# v51 LoRA Fine-tune — Qwen2.5-1.5B on T4

1. Upload `training/paraphrase_corpus.jsonl` to Colab (or mount Google Drive)
2. Run all cells (~30-90 min on T4)
3. Download `qwen2.5-1.5b-v51-q4_k_m.gguf` to `models/` locally
4. Set `LOCAL_MODEL_GGUF=qwen2.5-1.5b-v51-q4_k_m.gguf` and expand `LOCAL_ALLOWED_CATEGORIES` after holdout >=95%

In [ ]:
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CORPUS = '/content/drive/MyDrive/hybrid_routing/paraphrase_corpus.jsonl'  # adjust path
OUT_DIR = '/content/drive/MyDrive/hybrid_routing/v51_lora'

In [ ]:
import json
from datasets import Dataset

rows = [json.loads(l) for l in open(CORPUS) if l.strip() and json.loads(l).get('answer')]
print(f'Rows with gold answers: {len(rows)}')

def to_chat(row):
    cat = row['category']
    sys_map = {
        'summarization': 'Summarize per instructions. Output summary only.',
        'sentiment_classification': "Reply: '<Label> because <reason>.'",
        'factual_knowledge': 'Answer in plain prose. No markdown.',
        'named_entity_recognition': 'Output JSON entities only.',
    }
    system = sys_map.get(cat, 'Answer the query.')
    return {
        'messages': [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': row['prompt']},
            {'role': 'assistant', 'content': row['answer']},
        ]
    }

ds = Dataset.from_list([to_chat(r) for r in rows]).train_test_split(test_size=0.2, seed=42)
train_ds, eval_ds = ds['train'], ds['test']

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-1.5B-Instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)

def _to_text_batch(examples):
    convos = examples["messages"]
    texts = []
    for messages in convos:
        texts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        )
    return {"text": texts}

train_ds = train_ds.map(_to_text_batch, batched=True)
eval_ds = eval_ds.map(_to_text_batch, batched=True)
print("Sample text:", train_ds[0]["text"][:200], "...")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

def formatting_prompts_func(examples):
    texts = examples["text"]
    if isinstance(texts, str):
        return [texts]
    return list(texts)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    formatting_func=formatting_prompts_func,
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="no",  # avoid checkpoint pickle bug; merge saved in next cell
        output_dir=OUT_DIR,
        report_to="none",
    ),
)

trainer.train()
print("Training complete.")

In [ ]:
merged_dir = OUT_DIR + '/merged'
model.save_pretrained_merged(merged_dir, tokenizer, save_method='merged_16bit')
print('Merged model saved to', merged_dir)

In [ ]:
# Step A: HF -> GGUF f16 (skip if f16 already exists on Drive)
!git clone -q https://github.com/ggerganov/llama.cpp 2>/dev/null || true
!pip -q install sentencepiece

MERGED = "/content/drive/MyDrive/v51_lora/merged"
F16 = "/content/drive/MyDrive/qwen2.5-1.5b-v51-f16.gguf"
Q4  = "/content/drive/MyDrive/qwen2.5-1.5b-v51-q4_k_m.gguf"

import os
if not os.path.exists(F16):
    !python llama.cpp/convert_hf_to_gguf.py {MERGED} --outfile {F16} --outtype f16
else:
    print("f16 already exists, skipping convert")

# Step B: quantize with prebuilt ubuntu binary (no compile needed)
!curl -L -o /tmp/llama.zip https://github.com/ggml-org/llama.cpp/releases/download/b3620/llama-b3620-bin-ubuntu-x64.zip
!unzip -q -o /tmp/llama.zip -d /tmp/llama_bin
!/tmp/llama_bin/build/bin/llama-quantize {F16} {Q4} Q4_K_M

print("Done. Size MB:", round(os.path.getsize(Q4) / 1e6, 1))
print("Download this file from Drive:", Q4)

## GGUF export (run in separate cell or local machine)

```bash
git clone https://github.com/ggerganov/llama.cpp
pip install torch sentencepiece
python llama.cpp/convert_hf_to_gguf.py merged/ --outfile qwen2.5-1.5b-v51-f16.gguf
./llama.cpp/llama-quantize qwen2.5-1.5b-v51-f16.gguf qwen2.5-1.5b-v51-q4_k_m.gguf Q4_K_M
```

Copy `qwen2.5-1.5b-v51-q4_k_m.gguf` into repo `models/` and rebuild Docker with:
`ENV LOCAL_MODEL_GGUF=qwen2.5-1.5b-v51-q4_k_m.gguf`